# 01_simple_rag: End-to-End RAG Pipeline with Scraped Wikipedia Corpus

This notebook demonstrates an end-to-end RAG pipeline. It scrapes the Wikipedia page for "Retrieval-augmented generation", splits and indexes the paragraphs in a FAISS vector store, and executes a grounded query using OpenAI.


In [1]:
import os
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Load keys from root .env
load_dotenv(dotenv_path=r"d:\Study\Prep\.env")

# 2. Scrape Wikipedia Retrieval-Augmented Generation page
url = "https://en.wikipedia.org/wiki/Retrieval-augmented_generation"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
resp = requests.get(url, headers=headers)
soup = BeautifulSoup(resp.content, "html.parser")
# Extract non-empty paragraphs
paragraphs = [p.get_text().strip() for p in soup.find_all("p") if len(p.get_text().strip()) > 100]
corpus = paragraphs[:8]

print(f"Scraped {len(corpus)} paragraphs for indexing.")
print("Sample Paragraph 1:", corpus[0][:150] + "...")

# 3. Create FAISS vector store
embeddings = OpenAIEmbeddings()
db = FAISS.from_texts(corpus, embeddings)

# 4. Search query
query = "Who introduced retrieval-augmented generation and in which year?"
retrieved_docs = db.similarity_search(query, k=1)
retrieved_context = retrieved_docs[0].page_content

print("\nQuery:", query)
print("Retrieved Context:", retrieved_context[:200] + "...")

# 5. LLM Synthesis
prompt_tmpl = ChatPromptTemplate.from_template(
    "Use the context to answer the question.\nContext: {context}\nQuestion: {question}"
)
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
chain = prompt_tmpl | model | StrOutputParser()
completion = chain.invoke({"context": retrieved_context, "question": query})
print("\nLLM Completion:", completion)


C:\Users\aryan\AppData\Local\Temp\ipykernel_15832\2273971841.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Scraped 8 paragraphs for indexing.
Sample Paragraph 1: Retrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to retrieve and incorporate new information from externa...



Query: Who introduced retrieval-augmented generation and in which year?
Retrieved Context: The term retrieval-augmented generation (RAG) was introduced in a 2020 paper that described combining a parametric language model with a non-parametric external memory accessed through retrieval at in...



LLM Completion: Retrieval-augmented generation (RAG) was introduced in a 2020 paper.


### Output Explanation
- The crawler successfully fetches the RAG Wikipedia page and extracts paragraphs.
- FAISS embeds and indexes the text blocks.
- The similarity search retrieves the paragraph detailing that Lewis et al. from Facebook AI Research introduced RAG in 2020.
- The LLM completes the answer, grounded in the retrieved details.
